### Checking Capacities in Power Plant Matching Database

In [85]:
import pandas as pd
# Read powerplants.csv file

powerplants_df = pd.read_csv('powerplants.csv')

# display first 5 rows of the dataframe
powerplants_df.head()


,id,Name,Fueltype,Technology,Set,Country,Capacity,Efficiency,DateIn,DateRetrofit,DateOut,lat,lon,Duration,Volume_Mm3,DamHeight_m,StorageCapacity_MWh,EIC,projectID
0,0,Pumpspeicherkraftwerk Erzhausen,Hydro,Pumped Storage,Storage,Germany,200.0,0.75,1964.0,1998.0,NaN,51.898566,9.924887,5.510000,1.600000,287.0,1102.0,{nan},"{'MASTR': {'MASTR-SEE915985628661'}, 'GEM': {'..."
1,1,La Plate Taille,Hydro,Pumped Storage,Store,Belgium,144.0,NaN,1970.0,NaN,NaN,50.188400,4.386200,4.930556,68.400000,70.0,710.0,{nan},"{'GEM': {'G100000600151'}, 'JRC': {'JRC-H347'}..."
2,2,Illwerke Vkw Rodundwerk,Hydro,Reservoir,Store,Austria,495.0,0.75,1943.0,2011.0,NaN,47.085032,9.880116,588.383838,2.240000,353.0,291250.0,"{nan, nan, nan, nan, nan}","{'MASTR': {'MASTR-SEE952262880046', 'MASTR-SEE..."
3,3,Bissorte,Hydro,Pumped Storage,Store,France,818.0,NaN,1936.0,NaN,NaN,45.203600,6.581450,3.818182,39.500000,1160.0,3150.0,"{nan, nan}","{'GEM': {'G100001052026', 'G100000601707'}, 'J..."
4,4,Obervermuntwerk Maschine Turbine,Hydro,Pumped Storage,Storage,Austria,380.0,0.75,1943.0,2018.0,NaN,46.935290,10.059950,71.573684,35.630769,291.0,27198.0,"{nan, nan}","{'MASTR': {'MASTR-SEE926367113644', 'MASTR-SEE..."


In [86]:
print(powerplants_df['Fueltype'].unique())

['Hydro' 'Hard Coal' 'Natural Gas' 'Lignite' 'Oil' 'Wind' 'Solid Biomass'
 'Waste' 'Solar' 'Geothermal' 'Battery' 'Heat Storage' 'Nuclear' 'Other'
 'Biogas' 'Mechanical Storage' 'Hydrogen Storage']


In [87]:
# Find power plants in Denmark only
dk_all_power = powerplants_df[powerplants_df['Country'] == 'Denmark']

# Include plants active in 2025 or with no known decommissioning date
dk_all_power = dk_all_power[
    (dk_all_power['DateOut'] > 2025) |
    (dk_all_power['DateOut'].isna()) &
    (dk_all_power['DateIn'] < 2026) |
    (dk_all_power['DateIn'].isna())
]


In [88]:
# List all fuel types and tech in dk_all_power
print(dk_all_power['Fueltype'].unique())
print(dk_all_power['Technology'].unique())
print(dk_all_power['Set'].unique())


['Hard Coal' 'Solid Biomass' 'Oil' 'Natural Gas' 'Solar' 'Wind' 'Waste'
 'Battery' 'Hydrogen Storage' 'Heat Storage']
['Steam Turbine' nan 'CCGT' 'PV' 'Offshore' 'Csp' 'Onshore' 'Li'
 'Molten Salt' 'V']
['CHP' 'PP' nan 'Store']


In [89]:
dk_conv_power = dk_all_power[
    (dk_all_power['Fueltype'] == 'Hard Coal') |
    (dk_all_power['Fueltype'] == 'Solid Biomass') |
    (dk_all_power['Fueltype'] == 'Natural Gas') |
    (dk_all_power['Fueltype'] == 'Oil') |
    (dk_all_power['Fueltype'] == 'Waste')
]

dk_ren_power = dk_all_power[
    (dk_all_power['Fueltype'] == 'Wind') |
    (dk_all_power['Fueltype'] == 'Solar')
]

dk_store_power = dk_all_power[
    (dk_all_power['Fueltype'] == 'Battery')
]



In [90]:
tot_conv_power = dk_conv_power['Capacity'].sum()
tot_storage_power = dk_store_power['Capacity'].sum()
tot_ren_power = dk_ren_power['Capacity'].sum()


print(f'Hard Coal, Solid Biomass, Natural Gas, Oil and Waste Capacities Installed in Denmark 2025:', tot_conv_power, "MW")

print(f'Wind and Solar Capacities Installed in Denmark 2025', tot_ren_power, "MW")

print(f'Battery Storage Capacities Installed in Denmark 2025:', tot_storage_power, "MW")

print(f'Total Capacities Installed in Denmark 2025:', tot_conv_power + tot_ren_power + tot_storage_power, "MW")






Hard Coal, Solid Biomass, Natural Gas, Oil and Waste Capacities Installed in Denmark 2025: 3951.2 MW
Wind and Solar Capacities Installed in Denmark 2025 8073.200000000001 MW
Battery Storage Capacities Installed in Denmark 2025: 52.120000000000005 MW
Total Capacities Installed in Denmark 2025: 12076.520000000002 MW


## Checking Capacities defined from Energistyrelsen

### Wind and solar


In [91]:
import requests
import numpy as np
from shapely.geometry import shape

def postcode_to_coords(postcode):
    if pd.isna(postcode):
        return {
            "postcode": postcode,
            "longitude": np.nan,
            "latitude": np.nan
        }

    postcode = str(postcode).strip()

    url = f"https://api.dataforsyningen.dk/postnumre/{postcode}?format=geojson&landpostnumre"

    response = requests.get(url)

    if response.status_code != 200:
        print(f"Could not find postcode: {postcode}")
        return {
            "postcode": postcode,
            "longitude": np.nan,
            "latitude": np.nan
        }

    data = response.json()

    if "geometry" not in data:
        print(f"No geometry for postcode: {postcode}")
        return {
            "postcode": postcode,
            "longitude": np.nan,
            "latitude": np.nan
        }

    geometry = shape(data["geometry"])
    centroid = geometry.centroid

    return {
        "postcode": postcode,
        "longitude": centroid.x,
        "latitude": centroid.y
    }

In [92]:
# load vind_og_sol.csv file

df_wind_pv = pd.read_csv('vind_og_sol.csv', delimiter=';')


In [93]:
# filter the dataframe by status to be 'I drift'
df_wind_pv = df_wind_pv[df_wind_pv['Status'] == 'I drift']

In [94]:
df_wind_pv['Postnr.'] = (
    pd.to_numeric(df_wind_pv['Postnr.'], errors='coerce')
    .astype('Int64')
    .astype('string')
    .str.zfill(4)
)

In [95]:
postcodes = df_wind_pv['Postnr.'].dropna().unique()

In [96]:
postcode_coords = {
    postcode: postcode_to_coords(postcode)
    for postcode in postcodes
}

Could not find postcode: 9999
Could not find postcode: 9998
Could not find postcode: 0006
Could not find postcode: 1870


In [97]:
df_wind_pv['postcode_lon'] = df_wind_pv['Postnr.'].map(
    lambda p: postcode_coords.get(p, {}).get('longitude', np.nan)
)

df_wind_pv['postcode_lat'] = df_wind_pv['Postnr.'].map(
    lambda p: postcode_coords.get(p, {}).get('latitude', np.nan)
)

In [98]:
windfarm_coords = {
    "Middelgrundens Havvindmøllepark": (55.6923, 12.6708),
    "Horns Rev 2": (55.6024, 7.5902),
    "Rødsand 2": (54.5265, 11.6170),
    "Anholt havvindmøllepark 1": (56.6015, 11.2291),
    "Tunø Knob Vindmøllepark": (55.9693, 10.3553),
    "MIDDELGRUNDEN": (55.6923, 12.6708),
    "Rønland Havvindmøllepark 2": (56.6704, 8.2162),
    "Rødsand": (54.5254, 11.7597),
    "Horns Rev 1": (55.4882, 7.8407),
    "Rønland Havvindmøllepark 1": (56.6704, 8.2162),
    "Horns Rev 3": (55.6876, 7.6677),
    "Kriegers Flak A": (55.0194, 12.8299),
    "Kriegers Flak B": (55.0382, 12.9926),
    "Vesterhav Nord Mølle 1 til 21": (56.6999, 8.0446),
    "Vesterhav Syd TA31 Ringkøbing": (56.0329, 8.0267),
    "Nissum Bredning Vindpark": (56.6771, 8.2518),
}

# Start with postcode coordinates for all installations
df_wind_pv["lat"] = pd.to_numeric(
    df_wind_pv["postcode_lat"], errors="coerce"
)

df_wind_pv["lon"] = pd.to_numeric(
    df_wind_pv["postcode_lon"], errors="coerce"
)

# Clean names before matching
names = df_wind_pv["Navn"].astype("string").str.strip()

offshore_lat = names.map(
    lambda name: windfarm_coords.get(name, (np.nan, np.nan))[0]
)

offshore_lon = names.map(
    lambda name: windfarm_coords.get(name, (np.nan, np.nan))[1]
)

# Override only successfully matched offshore farms
matched_offshore = offshore_lat.notna() & offshore_lon.notna()

df_wind_pv.loc[matched_offshore, "lat"] = offshore_lat[matched_offshore]
df_wind_pv.loc[matched_offshore, "lon"] = offshore_lon[matched_offshore]

In [99]:
# make a new column 'Capacity' in df_wind_pv and set it to the value of 'InstalleretkW'

df_wind_pv['Capacity'] = df_wind_pv['InstalleretkW']/1000 # convert to MW

In [100]:
# Make a new column called Fueltype and set it to wind if 'Kategori' is not solcelle

df_wind_pv['Fueltype'] = df_wind_pv['Kategori'].apply(lambda x: 'Wind' if x != 'Solcelle' else 'Solar')

In [101]:
# make a new column called technology and set if fueltype is Wind and Placering is 'Hav' then set it to 'Offshore', if fueltype is Wind and Placering is 'Land' then set it to 'Onshore', if fueltype is Solar then set it to 'PV'

df_wind_pv['Technology'] = df_wind_pv.apply(lambda x: 'Offshore' if x['Fueltype'] == 'Wind' and x['Placering'] == 'HAV' else ('Onshore' if x['Fueltype'] == 'Wind' and x['Placering'] == 'LAND' else ('PV' if x['Fueltype'] == 'Solar' else np.nan)), axis=1)

In [102]:
# make new column called set and fill out with PP
df_wind_pv['Set'] = 'PP'

In [103]:
df_wind_pv_ppm = pd.DataFrame({
    "Name": (
        df_wind_pv["Navn"].fillna("")
        + " ("
        + df_wind_pv["Stamdata GSRN"].fillna("")
        + ")"
    ),

    "Fueltype": df_wind_pv["Fueltype"],
    "Technology": df_wind_pv["Technology"],
    "Set": df_wind_pv["Set"],
    "Country": "Denmark",
    "Capacity": df_wind_pv["Capacity"],
    "Efficiency": np.nan,

    "DateIn": pd.to_datetime(
        df_wind_pv["Idriftsdato"],
        errors="coerce"
    ).dt.year,

    "DateRetrofit": np.nan,

    "DateOut": pd.to_datetime(
        df_wind_pv["Afmeldt dato"],
        errors="coerce"
    ).dt.year,

    "lat": df_wind_pv["lat"],
    "lon": df_wind_pv["lon"],

    "Duration": np.nan,
    "Volume_Mm3": np.nan,
    "DamHeight_m": np.nan,
    "StorageCapacity_MWh": np.nan,
    "EIC": "{}",

    "projectID": "{}"
})

C:\Users\frede\AppData\Local\Temp\ipykernel_31084\570369923.py:16: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  "DateIn": pd.to_datetime(


In [104]:
df_wind_pv_ppm

,Name,Fueltype,Technology,Set,Country,Capacity,Efficiency,DateIn,DateRetrofit,DateOut,lat,lon,Duration,Volume_Mm3,DamHeight_m,StorageCapacity_MWh,EIC,projectID
1,Karlslunde (Greve) ('570714700000050084),Wind,Onshore,PP,Denmark,0.9000,NaN,1993,NaN,NaN,55.566551,12.221041,NaN,NaN,NaN,NaN,{},{}
2,Gøderup ('570714700000050091),Wind,Onshore,PP,Denmark,1.5000,NaN,1994,NaN,NaN,55.657020,12.082258,NaN,NaN,NaN,NaN,{},{}
3,Onsved ('570714700000050138),Wind,Onshore,PP,Denmark,1.2000,NaN,1999,NaN,NaN,55.763915,11.965541,NaN,NaN,NaN,NaN,{},{}
4,Havrebjerg ('570714700000050190),Wind,Onshore,PP,Denmark,0.6000,NaN,1991,NaN,NaN,55.405652,11.361094,NaN,NaN,NaN,NaN,{},{}
5,Fuglholmsgård Vindmølleklynge ('57071470000005...,Wind,Onshore,PP,Denmark,0.6000,NaN,1992,NaN,NaN,55.405652,11.361094,NaN,NaN,NaN,NaN,{},{}
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
184959,"Allegrovej 8A, 3650 Ølstykke ('571313179105239...",Solar,PV,PP,Denmark,0.0200,NaN,2025,NaN,NaN,55.791795,12.156178,NaN,NaN,NaN,NaN,{},{}
184960,"Skovvej 99, 2920 Charlottenlund ('571313179105...",Solar,PV,PP,Denmark,0.0114,NaN,2025,NaN,NaN,55.757515,12.573861,NaN,NaN,NaN,NaN,{},{}
184961,"Plantagevej 29A, 3480 Fredensborg ('5713131791...",Solar,PV,PP,Denmark,0.0108,NaN,2025,NaN,NaN,55.972154,12.398406,NaN,NaN,NaN,NaN,{},{}
184962,"Mellemsvinget 14, 3310 Ølsted ('57131317910524...",Solar,PV,PP,Denmark,0.0100,NaN,2025,NaN,NaN,55.915212,12.077552,NaN,NaN,NaN,NaN,{},{}


### Danish Power Plants 

In [105]:
# Load danish_power_plants.csv
df_dk_power_plants = pd.read_csv('danish_power_plants.csv', delimiter=';')

df_dk_power_plants


,aar,selskab_id,vaerk_id,vrkanl_ny,selskab_navn,vaerk_postnr,vaerk_postdistrikt,vaerk_kommune,vrktypenavn,fv_net,...,Naturgas_andel,Affald_andel,Biogas_andel,Fast biomasse_andel,Bio-olie_andel,Braendselsfrit_andel,Solenergi_andel,Vandkraft_andel,Elektricitet_andel,Omgivelsesvarme_andel
0,2025,1.0,1.0,1-14,Crossbridge Energy A/S,7000.0,Fredericia,607,Erhvervsværk,81.0,...,NaN,NaN,NaN,0,NaN,1,NaN,NaN,NaN,NaN
1,2025,1.0,1.0,1-2,Crossbridge Energy A/S,7000.0,Fredericia,607,Erhvervsværk,81.0,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
2,2025,4.0,3.0,3-2,ASSENS FJERNVARME A M B A,9550.0,Mariager,846,Fjernvarmeværk,209.0,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
3,2025,4.0,3.0,3-3,ASSENS FJERNVARME A M B A,9550.0,Mariager,846,Fjernvarmeværk,209.0,...,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN
4,2025,5.0,5.0,5-2,AULUM FJERNVARME AMBA,7490.0,Aulum,657,Decentralt værk,159.0,...,1,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2911,2025,NaN,NaN,NaN,NaN,NaN,NaN,580,Erhvervsværk,NaN,...,NaN,NaN,1,0,NaN,NaN,NaN,NaN,NaN,NaN
2912,2025,NaN,NaN,NaN,NaN,NaN,NaN,840,Fjernvarmeværk,NaN,...,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN
2913,2025,NaN,NaN,NaN,NaN,NaN,NaN,840,Fjernvarmeværk,NaN,...,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN
2914,2025,NaN,NaN,NaN,NaN,NaN,NaN,461,Erhvervsværk,NaN,...,1,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN


In [106]:
# New dataframe only with the columns we need

df_dk_power_plants = df_dk_power_plants[['vrkanl_ny', 
                                         'fv_net_navn',
                                         'vaerk_postnr',
                                         'anlaegstype_navn',
                                         'idriftdato',
                                         'skrotdato',
                                         'elkapacitet_MW',
                                         'varmekapacitet_MW',
                                         'Hovedbrændselsgruppe' ]]




In [107]:
df_dk_power_plants['Hovedbrændselsgruppe'].unique()

array(['braendselsfrit', 'olie', 'fast biomasse', 'naturgas',
       'elektricitet', 'solenergi', 'omgivelsesvarme',
       'Ej i drift i 2025', 'biogas', 'bio-olie', 'affald', 'vandkraft',
       'kul'], dtype=object)

In [108]:
# Postcode
df_dk_power_plants['vaerk_postnr'] = (
    pd.to_numeric(df_dk_power_plants['vaerk_postnr'], errors='coerce')
    .astype('Int64')
    .astype('string')
    .str.zfill(4)
)

# Electricity capacity
df_dk_power_plants['elkapacitet_MW'] = pd.to_numeric(
    df_dk_power_plants['elkapacitet_MW']
    .astype(str)
    .str.replace(',', '.', regex=False),
    errors='coerce'
)

# Heat capacity
df_dk_power_plants['varmekapacitet_MW'] = pd.to_numeric(
    df_dk_power_plants['varmekapacitet_MW']
    .astype(str)
    .str.replace(',', '.', regex=False),
    errors='coerce'
)

C:\Users\frede\AppData\Local\Temp\ipykernel_31084\107067923.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_dk_power_plants['vaerk_postnr'] = (
C:\Users\frede\AppData\Local\Temp\ipykernel_31084\107067923.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_dk_power_plants['elkapacitet_MW'] = pd.to_numeric(
C:\Users\frede\AppData\Local\Temp\ipykernel_31084\107067923.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_inde

In [109]:
# remove rows with 'Ej i drift i 2025' in 'Hovedbrændselgruppe' column
df_dk_power_plants = df_dk_power_plants[df_dk_power_plants['Hovedbrændselsgruppe'] != 'Ej i drift i 2025']

# add new column 'Set' 
df_dk_power_plants.insert(0, 'Set', 'DK')




In [110]:
# make sure that set matches powerplantmatching format

df_dk_power_plants.loc[
    (df_dk_power_plants['varmekapacitet_MW'] > 0) &
    (df_dk_power_plants['elkapacitet_MW'] > 0),
    'Set'
] = 'CHP'

df_dk_power_plants.loc[
    (df_dk_power_plants['varmekapacitet_MW'] == 0) &
    (df_dk_power_plants['elkapacitet_MW'] > 0),
    'Set'
] = 'PP'

In [111]:
# remove all where elkapacitet_MW is 0, only want PP and CHP

df_dk_power_plants = df_dk_power_plants[df_dk_power_plants['elkapacitet_MW'] > 0]

In [112]:
# remove all rows where vaerk_postnr is NaN

df_dk_power_plants = df_dk_power_plants[df_dk_power_plants['vaerk_postnr'].notna()]

In [113]:

# map technologies to PPM format

technology_mapping = {
    "Gasturbine": "OCGT",
    "Forbrændingsmotor": "Combustion Engine",
    "Dampturbine": "Steam Turbine",
    "Kombianlæg": "CCGT",
    "Bioforgasn. m. FM": "Combustion Engine",
    "Vandkraft": "Run-Of-River",
    "Nødstrømsanlæg": "Combustion Engine",
    "Organic Rankine (ORC)": None,
}

df_dk_power_plants["Technology"] = (
    df_dk_power_plants["anlaegstype_navn"].map(technology_mapping)
)

In [114]:
# map fueltypes to PPM format
fueltype_mapping = {
    "olie": "Oil",
    "naturgas": "Natural Gas",
    "fast biomasse": "Solid Biomass",
    "biogas": "Biogas",
    "affald": "Waste",
    "vandkraft": 'Hydro',
    "kul": "Hard Coal",
}

df_dk_power_plants["Fueltype_PPM"] = (
    df_dk_power_plants["Hovedbrændselsgruppe"].map(fueltype_mapping)
)

In [115]:
# change vaerk_postnr to two columns, longitude and latitude using postcode_to_coords function

df_dk_power_plants[['vaerk_postnr', 'longitude', 'latitude']] = (
    
        df_dk_power_plants['vaerk_postnr']
        .apply(postcode_to_coords)
        .apply(pd.Series)
    )


In [116]:
df_dk_power_plants

,Set,vrkanl_ny,fv_net_navn,vaerk_postnr,anlaegstype_navn,idriftdato,skrotdato,elkapacitet_MW,varmekapacitet_MW,Hovedbrændselsgruppe,Technology,Fueltype_PPM,longitude,latitude
1,CHP,1-2,TVIS,7000,Gasturbine,1993-01-01 00:00:00,NaN,26.000,52.000,olie,OCGT,Oil,9.659946,55.582883
4,CHP,5-2,Aulum Fjernvarme,7490,Forbrændingsmotor,2006-12-12 00:00:00,NaN,5.100,5.650,naturgas,Combustion Engine,Natural Gas,8.819162,56.284991
15,CHP,9-5,Agersted Fjernvarme,9330,Forbrændingsmotor,2005-09-20 00:00:00,NaN,1.410,1.900,naturgas,Combustion Engine,Natural Gas,10.304552,57.190222
33,CHP,1942-2,Egtved Fjernvarme,6040,Forbrændingsmotor,1995-01-01 00:00:00,NaN,3.730,4.220,naturgas,Combustion Engine,Natural Gas,9.343440,55.608385
39,CHP,15-1,Flauenskjold Fjernvarme,9330,Forbrændingsmotor,1993-01-01 00:00:00,NaN,0.730,0.960,naturgas,Combustion Engine,Natural Gas,10.304552,57.190222
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2820,CHP,1211-6,NaN,9700,Forbrændingsmotor,2012-01-01 00:00:00,NaN,0.340,0.520,biogas,Combustion Engine,Biogas,9.926182,57.271516
2821,CHP,1211-7,NaN,9700,Forbrændingsmotor,2013-01-01 00:00:00,NaN,0.730,1.000,biogas,Combustion Engine,Biogas,9.926182,57.271516
2830,CHP,1091-11,Videbæk Fjernvarme,6920,Forbrændingsmotor,2019-05-07 00:00:00,NaN,3.047,4.002,biogas,Combustion Engine,Biogas,8.680097,56.074491
2831,CHP,1091-12,Videbæk Fjernvarme,6920,Forbrændingsmotor,2019-05-07 00:00:00,NaN,3.047,4.002,biogas,Combustion Engine,Biogas,8.680097,56.074491


In [117]:
# sum elkapacitet_MW
total_el_capacity = df_dk_power_plants['elkapacitet_MW'].sum()

print(total_el_capacity)

5263.833999999999


In [118]:
# make a new dataframe with the following columns 

df_ppm = pd.read_csv('powerplants.csv')




In [119]:
# remove all plants located in denmark
df_ppm = df_ppm[df_ppm['Country'] != 'Denmark']


In [120]:
df_ppm[df_ppm["Country"] == "Denmark"].head()

,id,Name,Fueltype,Technology,Set,Country,Capacity,Efficiency,DateIn,DateRetrofit,DateOut,lat,lon,Duration,Volume_Mm3,DamHeight_m,StorageCapacity_MWh,EIC,projectID


In [121]:
import numpy as np

df_dk_ppm = pd.DataFrame({
    "Name": (
        df_dk_power_plants["fv_net_navn"].fillna("")
        + " ("
        + df_dk_power_plants["vrkanl_ny"].fillna("")
        + ")"
    ),

    "Fueltype": df_dk_power_plants["Fueltype_PPM"],
    "Technology": df_dk_power_plants["Technology"],
    "Set": df_dk_power_plants["Set"],
    "Country": "Denmark",
    "Capacity": df_dk_power_plants["elkapacitet_MW"],
    "Efficiency": np.nan,

    "DateIn": pd.to_datetime(
        df_dk_power_plants["idriftdato"],
        errors="coerce"
    ).dt.year,

    "DateRetrofit": np.nan,

    "DateOut": pd.to_datetime(
        df_dk_power_plants["skrotdato"],
        errors="coerce"
    ).dt.year,

    "lat": df_dk_power_plants["latitude"],
    "lon": df_dk_power_plants["longitude"],

    "Duration": np.nan,
    "Volume_Mm3": np.nan,
    "DamHeight_m": np.nan,
    "StorageCapacity_MWh": np.nan,
    "EIC": "{}",

    "projectID": "{}"
})

In [122]:
df_ppm = pd.concat(
    [df_ppm, df_dk_ppm, df_wind_pv_ppm],
    ignore_index=True
)

In [123]:
# id need to match row number

df_ppm['id'] = df_ppm.index


In [124]:
df_ppm

,id,Name,Fueltype,Technology,Set,Country,Capacity,Efficiency,DateIn,DateRetrofit,DateOut,lat,lon,Duration,Volume_Mm3,DamHeight_m,StorageCapacity_MWh,EIC,projectID
0,0,Pumpspeicherkraftwerk Erzhausen,Hydro,Pumped Storage,Storage,Germany,200.0000,0.75,1964.0,1998.0,NaN,51.898566,9.924887,5.510000,1.600000,287.0,1102.0,{nan},"{'MASTR': {'MASTR-SEE915985628661'}, 'GEM': {'..."
1,1,La Plate Taille,Hydro,Pumped Storage,Store,Belgium,144.0000,NaN,1970.0,NaN,NaN,50.188400,4.386200,4.930556,68.400000,70.0,710.0,{nan},"{'GEM': {'G100000600151'}, 'JRC': {'JRC-H347'}..."
2,2,Illwerke Vkw Rodundwerk,Hydro,Reservoir,Store,Austria,495.0000,0.75,1943.0,2011.0,NaN,47.085032,9.880116,588.383838,2.240000,353.0,291250.0,"{nan, nan, nan, nan, nan}","{'MASTR': {'MASTR-SEE952262880046', 'MASTR-SEE..."
3,3,Bissorte,Hydro,Pumped Storage,Store,France,818.0000,NaN,1936.0,NaN,NaN,45.203600,6.581450,3.818182,39.500000,1160.0,3150.0,"{nan, nan}","{'GEM': {'G100001052026', 'G100000601707'}, 'J..."
4,4,Obervermuntwerk Maschine Turbine,Hydro,Pumped Storage,Storage,Austria,380.0000,0.75,1943.0,2018.0,NaN,46.935290,10.059950,71.573684,35.630769,291.0,27198.0,"{nan, nan}","{'MASTR': {'MASTR-SEE926367113644', 'MASTR-SEE..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
345038,345038,"Allegrovej 8A, 3650 Ølstykke ('571313179105239...",Solar,PV,PP,Denmark,0.0200,NaN,2025.0,NaN,NaN,55.791795,12.156178,NaN,NaN,NaN,NaN,{},{}
345039,345039,"Skovvej 99, 2920 Charlottenlund ('571313179105...",Solar,PV,PP,Denmark,0.0114,NaN,2025.0,NaN,NaN,55.757515,12.573861,NaN,NaN,NaN,NaN,{},{}
345040,345040,"Plantagevej 29A, 3480 Fredensborg ('5713131791...",Solar,PV,PP,Denmark,0.0108,NaN,2025.0,NaN,NaN,55.972154,12.398406,NaN,NaN,NaN,NaN,{},{}
345041,345041,"Mellemsvinget 14, 3310 Ølsted ('57131317910524...",Solar,PV,PP,Denmark,0.0100,NaN,2025.0,NaN,NaN,55.915212,12.077552,NaN,NaN,NaN,NaN,{},{}


In [125]:
df_ppm[df_ppm["Country"] == "Denmark"]

,id,Name,Fueltype,Technology,Set,Country,Capacity,Efficiency,DateIn,DateRetrofit,DateOut,lat,lon,Duration,Volume_Mm3,DamHeight_m,StorageCapacity_MWh,EIC,projectID
164696,164696,TVIS (1-2),Oil,OCGT,CHP,Denmark,26.0000,NaN,1993.0,NaN,NaN,55.582883,9.659946,NaN,NaN,NaN,NaN,{},{}
164697,164697,Aulum Fjernvarme (5-2),Natural Gas,Combustion Engine,CHP,Denmark,5.1000,NaN,2006.0,NaN,NaN,56.284991,8.819162,NaN,NaN,NaN,NaN,{},{}
164698,164698,Agersted Fjernvarme (9-5),Natural Gas,Combustion Engine,CHP,Denmark,1.4100,NaN,2005.0,NaN,NaN,57.190222,10.304552,NaN,NaN,NaN,NaN,{},{}
164699,164699,Egtved Fjernvarme (1942-2),Natural Gas,Combustion Engine,CHP,Denmark,3.7300,NaN,1995.0,NaN,NaN,55.608385,9.343440,NaN,NaN,NaN,NaN,{},{}
164700,164700,Flauenskjold Fjernvarme (15-1),Natural Gas,Combustion Engine,CHP,Denmark,0.7300,NaN,1993.0,NaN,NaN,57.190222,10.304552,NaN,NaN,NaN,NaN,{},{}
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
345038,345038,"Allegrovej 8A, 3650 Ølstykke ('571313179105239...",Solar,PV,PP,Denmark,0.0200,NaN,2025.0,NaN,NaN,55.791795,12.156178,NaN,NaN,NaN,NaN,{},{}
345039,345039,"Skovvej 99, 2920 Charlottenlund ('571313179105...",Solar,PV,PP,Denmark,0.0114,NaN,2025.0,NaN,NaN,55.757515,12.573861,NaN,NaN,NaN,NaN,{},{}
345040,345040,"Plantagevej 29A, 3480 Fredensborg ('5713131791...",Solar,PV,PP,Denmark,0.0108,NaN,2025.0,NaN,NaN,55.972154,12.398406,NaN,NaN,NaN,NaN,{},{}
345041,345041,"Mellemsvinget 14, 3310 Ølsted ('57131317910524...",Solar,PV,PP,Denmark,0.0100,NaN,2025.0,NaN,NaN,55.915212,12.077552,NaN,NaN,NaN,NaN,{},{}


In [126]:
# save as powerplants.csv in another folder
df_ppm.to_csv('powerplants_updated.csv', index=False)
